# 01 — TMDB Poster Scraping
Hedef: 10.000–12.000 film afişi + `labels.csv`

In [ ]:
# --- Google Colab: Drive bağlantısı ---
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT  = '/content/drive/MyDrive/film-genre-project'   # kod (git repo)
# DATA_ROOT  = '/content/drive/MyDrive/film-genre-project-data'  # veri

# --- Lokalde çalıştırma ---
from pathlib import Path
CODE_ROOT = Path('..').resolve()
DATA_ROOT = CODE_ROOT  # lokalde proje klasörünün kendisi

POSTERS_DIR  = DATA_ROOT / 'posters'
LABELS_PATH  = DATA_ROOT / 'labels.csv'

POSTERS_DIR.mkdir(exist_ok=True)
print('Kod:', CODE_ROOT)
print('Veri:', DATA_ROOT)

In [ ]:
import subprocess

SCRAPY_DIR = CODE_ROOT / 'src' / 'scraper'
SCRAPY_BIN = CODE_ROOT / '.venv' / 'Scripts' / 'scrapy.exe'

# Colab'da:
# !pip install scrapy requests pillow -q
# SCRAPY_BIN = Path('scrapy')  # PATH'ten

print('Scrapy dir:', SCRAPY_DIR)
print('Scrapy bin:', SCRAPY_BIN)

## 100 Filmlik Test Çekimi

In [ ]:
result = subprocess.run(
    [str(SCRAPY_BIN), 'crawl', 'tmdb',
     '-s', 'CLOSESPIDER_ITEMCOUNT=100',
     '-s', 'max_pages=10',
     '-s', f'IMAGES_STORE={POSTERS_DIR}',
     '-s', f'LABELS_PATH={LABELS_PATH}',
     '-L', 'INFO'],
    cwd=str(SCRAPY_DIR),
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
import pandas as pd

df = pd.read_csv(LABELS_PATH)
print(f'Toplam film: {len(df)}')
df.head(10)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

all_genres = [g for genres in df['genres'].str.split('|') for g in genres]
genre_counts = Counter(all_genres)

genres_df = pd.DataFrame(genre_counts.most_common(), columns=['genre', 'count'])
print(genres_df.to_string())

plt.figure(figsize=(12, 5))
plt.bar(genres_df['genre'], genres_df['count'])
plt.xticks(rotation=45, ha='right')
plt.title('Tür Dağılımı (Test Çekimi)')
plt.tight_layout()
plt.show()

## Tam Çekim (10.000–12.000 Film)
Test çekimi başarılıysa aşağıdaki hücreyi çalıştır.

In [ ]:
# Önceki test verilerini temizlemek istersen:
# import shutil
# shutil.rmtree(POSTERS_DIR, ignore_errors=True)
# LABELS_PATH.unlink(missing_ok=True)

result = subprocess.run(
    [str(SCRAPY_BIN), 'crawl', 'tmdb',
     '-s', f'IMAGES_STORE={POSTERS_DIR}',
     '-s', f'LABELS_PATH={LABELS_PATH}',
     '-L', 'WARNING'],
    cwd=str(SCRAPY_DIR),
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# Çekim sonrası özet
poster_count = len(list(POSTERS_DIR.glob('*.jpg')))
df_final = pd.read_csv(LABELS_PATH)

print(f'İndirilen poster: {poster_count}')
print(f'labels.csv satır sayısı: {len(df_final)}')

all_genres = [g for genres in df_final['genres'].str.split('|') for g in genres]
genre_counts = Counter(all_genres)
print('\nTür dağılımı:')
for genre, count in sorted(genre_counts.items(), key=lambda x: -x[1]):
    status = '✓' if count >= 500 else '✗ (<500)'
    print(f'  {status} {genre}: {count}')